In [8]:
import numpy as np
import pandas as pd

import re
import orjson

from gensim.models import FastText

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    recall_score,
    precision_score,
    average_precision_score
)

from hydra import compose, initialize_config_dir
from omegaconf import OmegaConf
from match import CONFIG_DIR, resolve_project_path, normalize_attributes 

In [54]:
with initialize_config_dir(version_base=None, config_dir=str(CONFIG_DIR)):
    cfg = compose(config_name="maxpooling_finalka")
cfg

{'dataset': {'items_human': 'data/items_human.parquet', 'matches_human': 'data/matches.parquet', 'items_human_normalized': 'data/items_human_normalized.parquet', 'synonyms': 'data/synonyms_df.parquet', 'unique_attributes': 'data/untouched_stats.parquet', 'fasttext_model_path': 'data/ft_vectors.kv'}}

# 1. Загрузка данных

In [26]:
items_human = pd.read_parquet(resolve_project_path(cfg.dataset.items_human_normalized))
matches = pd.read_parquet(resolve_project_path(cfg.dataset.matches_human))

In [27]:
items_human.head(5)

,id,name,attributes,category,normalized_attributes
0,197,victor reinz прокладка впускного коллектора ар...,"{""артикул"":""703315700"",""бренд"":""victor reinz"",...",Автотовары,"{""артикул"": ""703315700"", ""бренд"": ""victor rein..."
1,415,"stellox диск тормозной, арт. 8500892sx","{""артикул"":""stellox_8500892sx"",""место установк...",Автотовары,"{""артикул"": ""stellox_8500892sx"", ""система уста..."
2,427,комплект подшипника ступицы колеса lynxauto,"{""артикул производителя"":"""",""бренд"":""lynxauto""...",Автотовары,"{""артикул производителя"": """", ""бренд"": ""lynxau..."
3,1027,"kraft подшипник ступицы, арт. kt204632, 1 шт.","{""артикул"":""112097-01"",""бренд"":""kraft"",""количе...",Автотовары,"{""артикул"": ""112097-01"", ""бренд"": ""kraft"", ""ко..."
4,2639,фильтр салонный skoda fabia 00-,"{""альтернативные артикулы товара"":""8104400xkz9...",Автотовары,"{""альтернативные артикулы товара"": ""8104400xkz..."


In [28]:
matches.head(5)

,id1,id2,target
0,32,188978721454,0.0
1,34,197568543106,0.0
2,37,661425040668,0.0
3,97,274877917548,0.0
4,154,283467899529,0.0


# 2. Нормализация данных

In [25]:
non_alnum_re = re.compile(r'[^а-яёa-z0-9]+')
space_re = re.compile(r'\s+')

def normalize_text(text):
    text = str(text).lower()
    text = non_alnum_re.sub(' ', text)
    return space_re.sub(' ', text).strip()

# 3. Очистка атрибутов

In [29]:
# просто из json в dict
def parse_attributes(attrs):
    if attrs is None:
        return {}
    attrs = orjson.loads(attrs)
    return attrs

In [30]:
def prepare_attributes(attrs):
    attrs = parse_attributes(attrs)
    return {normalize_text(key): normalize_text(val) for key, val in attrs.items()}

# 4. Корпус для обучения FastText

In [31]:
# словарь атрибутов товара -> в список слов для обучения
def attributes_to_tokens(attrs):
    tokens = []
    
    for key, val in attrs.items():
        tokens.extend(key.split())
        tokens.extend(val.split())
    
    return tokens

In [48]:
def build_corpus(df):
    corpus = []

    for raw_attrs in df['attributes'].dropna():
        attrs = prepare_attributes(raw_attrs)
        tokens = attributes_to_tokens(attrs)
            
        if tokens:
            corpus.append(tokens)
    return corpus

In [49]:
corpus = build_corpus(items_human)

print(f"Длина корпуса: {len(corpus)}")
print(f"Сколько атрибутов пропустили: {len(items_human) - len(corpus)}")

Длина корпуса: 711232
Сколько атрибутов пропустили: 72


# 5. Обучение FastText

In [50]:
VECTOR_SIZE = 256
ft_model = FastText(
    vector_size=VECTOR_SIZE,
    window=5,
    min_count=2,
    workers=8,
    sg=1,
    min_n=2,
    max_n=5,
    epochs=10
)

In [51]:
ft_model.build_vocab(corpus_iterable=corpus)

In [52]:
ft_model.train(corpus, total_examples=len(corpus), epochs=ft_model.epochs)

(330478162, 400135870)

In [56]:
ft_model.wv.save(str(resolve_project_path(cfg.dataset.fasttext_model_path)))

# 6. Эмбединги атрибутов

In [57]:
def text_embedding(text, model):

    # если текст пустой, то возвращаем вектор из 0
    if not text:
        return np.zeros(model.vector_size, dtype=np.float32)

    tokens = text.split()

    # итоговый вектор = сумма векторов всех токенов / количество токенов
    embedding = np.zeros(model.vector_size, dtype=np.float32)

    for token in tokens:
        idx = model.wv.key_to_index.get(token)
        if idx is None:
            embedding += model.wv.get_vector(token)
        else:
            embedding += model.wv.vectors[idx]
    return embedding * np.float32(1.0/len(tokens))

In [58]:
def get_atribute_min_max(attrs, model):
    # возвращаем кортеж: название атрибута, 
    keys, vals, names = [], [], []

    for key, val in attrs.items():
        # считаем эмбединги
        key_emb = text_embedding(key, model)
        val_emb = text_embedding(val, model)

        # нормализуем эмбединги
        key_norm = np.linalg.norm(key_emb)
        val_norm = np.linalg.norm(val_emb)

        if key_norm > 0:
            key_emb = key_emb / key_norm
        if val_norm > 0:
            val_emb = val_emb / val_norm

        keys.append(key_emb)
        vals.append(val_emb)
        
        names.append(key)
    
    if not keys:
        empty = np.zeros(model.vector_size)
        return (names, empty, empty, empty, empty)
        
    keys = np.asarray(keys)
    vals = np.asarray(vals)

    max_keys = keys.max(axis=0)
    max_vals = vals.max(axis=0)
    min_keys = keys.min(axis=0)
    min_vals = vals.min(axis=0)
    
    return (names, min_keys, max_keys, min_vals, max_vals)

# 7. Создание кэша карточек

In [59]:
def max_pairwise_product(out, a_min, a_max, b_min, b_max, tmp):
    np.multiply(a_min, b_min, out=out)
    np.multiply(a_min, b_max, out=tmp)
    np.maximum(out, tmp, out=out)
    np.multiply(a_max, b_min, out=tmp)
    np.maximum(out, tmp, out=out)
    np.multiply(a_max, b_max, out=tmp)
    np.maximum(out, tmp, out=out)

In [60]:
def pair_embedding(out, card1, card2, tmp):
    d = len(card1[0])

    max_pairwise_product(out[:d], card1[0], card1[1], card2[0], card2[1], tmp[:d])
    max_pairwise_product(out[d:], card1[2], card1[3], card2[2], card2[3], tmp[d:])

In [61]:
def build_card_cache(df, model):
    ids = df['id']
    raw_attrs = df['attributes']
    cache = {}
    
    for item_id, raw_attr in zip(ids, raw_attrs):
        attrs = prepare_attributes(raw_attr)
        names, min_keys, max_keys, min_vals, max_vals = get_atribute_min_max(attrs, ft_model)

        cache[item_id] = [min_keys, max_keys, min_vals, max_vals]
        

    return cache

In [62]:
import time

start_time = time.perf_counter()
human_cache = build_card_cache(items_human, ft_model)

end_time = time.perf_counter()
execution_time = end_time - start_time
print(f"Code took {execution_time:.4f} seconds to complete.")

Code took 232.6059 seconds to complete.


# 8. Построение датасета

In [63]:
feature_dim = 2 * ft_model.vector_size

X = []
y = []
ids = {'id1' : [], 'id2': []}
skipped = 0

start_time = time.perf_counter()
for id1, id2, target in matches[['id1', 'id2', 'target']].itertuples(index=False, name=None):
    card1 = human_cache[id1]
    card2 = human_cache[id2]

    if card1 is None or card2 is None:
        skipped += 1
        continue
    out = np.zeros(feature_dim, dtype=np.float32)
    tmp = np.zeros(feature_dim, dtype=np.float32)

    pair_embedding(out, card1, card2, tmp)
    if out is not np.nan:
        X.append(out)
        y.append(target)
        ids['id1'].append(id1)
        ids['id2'].append(id2)

X = np.asarray(X)
y = np.asarray(y)

end_time = time.perf_counter()
execution_time = end_time - start_time
print(f"Code took {execution_time:.4f} seconds to complete.")

print("X shape:", X.shape)
print("y shape:", y.shape)

print("skipped:", skipped)

print("\nClass distribution:")
print(pd.Series(y).value_counts())
    

Code took 5.8317 seconds to complete.
X shape: (365654, 512)
y shape: (365654,)
skipped: 0

Class distribution:
0.0    271764
1.0     93890
Name: count, dtype: int64


In [64]:
ids = pd.DataFrame(ids)

In [65]:
X_train, X_test, y_train, y_test, ids_train, ids_test = train_test_split(X,y,ids,test_size=0.2,random_state=42,stratify=y)

print("train:", X_train.shape)
print("test:", X_test.shape)

train: (292523, 512)
test: (73131, 512)


# 9. Тест на простенькой MLP

In [66]:
import torch
import torch.nn as nn

from torch.utils.data import (
    TensorDataset,
    DataLoader
)

In [67]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

Device: cpu


In [68]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(
    X_train
)

X_test_scaled = scaler.transform(
    X_test
)

In [69]:
X_train_tensor = torch.tensor(
    X_train_scaled,
    dtype=torch.float32
)

y_train_tensor = torch.tensor(
    y_train,
    dtype=torch.float32
)

X_test_tensor = torch.tensor(
    X_test_scaled,
    dtype=torch.float32
)

y_test_tensor = torch.tensor(
    y_test,
    dtype=torch.float32
)

In [70]:
train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

test_dataset = TensorDataset(
    X_test_tensor,
    y_test_tensor
)

In [71]:
BATCH_SIZE = 512

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

## 9.1 Архитектура модели

In [72]:
class PairMLP(nn.Module):

    def __init__(self, input_dim, dropout=0.2):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim,256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256,128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128,64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64,1)
        )

    def forward(self, x):
        return self.network(x).squeeze(1)

In [73]:
INPUT_DIM = X_train.shape[1]

model = PairMLP(input_dim=INPUT_DIM, dropout=0.2).to(device)

In [74]:
num_positive = np.sum(y_train == 1)
num_negative = np.sum(y_train == 0)

In [75]:
pos_weight = (num_negative/max(num_positive, 1))

In [76]:
pos_weight_tensor = torch.tensor(
    [pos_weight],
    dtype=torch.float32,
    device=device
)

criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight_tensor
)

In [77]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

In [78]:
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=3
)

In [79]:
EPOCHS = 50
PATIENCE = 7
best_auc = -np.inf
best_state = None
epochs_without_improvement = 0
history = []

## 9.2 Обучение

In [80]:
def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    device
):

    model.train()

    total_loss = 0.0
    total_samples = 0

    for X_batch, y_batch in loader:
        
        X_batch = X_batch.to(
            device,
            non_blocking=True
        )

        y_batch = y_batch.to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad()

        logits = model(
            X_batch
        )
        loss = criterion(
            logits,
            y_batch
        )

        loss.backward()

        optimizer.step()

        batch_size = X_batch.size(0)

        total_loss += (
            loss.item()
            * batch_size
        )

        total_samples += batch_size

    return (
        total_loss
        /
        total_samples
    )

In [81]:
@torch.no_grad()
def evaluate(
    model,
    loader,
    criterion,
    device
):

    model.eval()

    total_loss = 0.0
    total_samples = 0

    all_probs = []
    all_targets = []

    for X_batch, y_batch in loader:

        X_batch = X_batch.to(
            device,
            non_blocking=True
        )

        y_batch = y_batch.to(
            device,
            non_blocking=True
        )
        logits = model(X_batch)

        loss = criterion(
            logits,
            y_batch
        )

        probs = torch.sigmoid(
            logits
        )

        batch_size = X_batch.size(0)

        total_loss += (
            loss.item()
            * batch_size
        )

        total_samples += batch_size

        all_probs.append(
            probs.cpu().numpy()
        )

        all_targets.append(
            y_batch.cpu().numpy()
        )

    all_probs = np.concatenate(
        all_probs
    )

    all_targets = np.concatenate(
        all_targets
    )

    avg_loss = (
        total_loss
        /
        total_samples
    )
    auc = roc_auc_score(
        all_targets,
        all_probs
    )

    

    return (
        avg_loss,
        auc,
        all_targets,
        all_probs
    )

In [82]:
import copy

In [83]:
for epoch in range(1, EPOCHS + 1):

    train_loss = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device
    )

    val_loss, val_auc, _, _ = evaluate(
        model,
        test_loader,
        criterion,
        device
    )

    scheduler.step(
        val_auc
    )

    current_lr = optimizer.param_groups[0]["lr"]

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_auc": val_auc,
        "lr": current_lr
    })

    print(
        f"Epoch {epoch:03d} | "
        f"train_loss={train_loss:.4f} | "
        f"val_loss={val_loss:.4f} | "
        f"val_auc={val_auc:.4f} | "
        f"lr={current_lr:.6f}"
    )

    # ---------------------------
    # Early stopping
    # ---------------------------

    if val_auc > best_auc:

        best_auc = val_auc

        best_state = copy.deepcopy(
            model.state_dict()
        )

        epochs_without_improvement = 0

    else:

        epochs_without_improvement += 1

        if (
            epochs_without_improvement
            >= PATIENCE
        ):

            print(
                "\nEarly stopping."
            )

            break

Epoch 001 | train_loss=0.8752 | val_loss=0.8513 | val_auc=0.7657 | lr=0.001000
Epoch 002 | train_loss=0.8501 | val_loss=0.8425 | val_auc=0.7719 | lr=0.001000
Epoch 003 | train_loss=0.8400 | val_loss=0.8360 | val_auc=0.7759 | lr=0.001000
Epoch 004 | train_loss=0.8307 | val_loss=0.8381 | val_auc=0.7745 | lr=0.001000
Epoch 005 | train_loss=0.8230 | val_loss=0.8337 | val_auc=0.7767 | lr=0.001000
Epoch 006 | train_loss=0.8149 | val_loss=0.8330 | val_auc=0.7779 | lr=0.001000
Epoch 007 | train_loss=0.8079 | val_loss=0.8319 | val_auc=0.7799 | lr=0.001000
Epoch 008 | train_loss=0.7984 | val_loss=0.8325 | val_auc=0.7790 | lr=0.001000
Epoch 009 | train_loss=0.7909 | val_loss=0.8314 | val_auc=0.7804 | lr=0.001000
Epoch 010 | train_loss=0.7821 | val_loss=0.8440 | val_auc=0.7769 | lr=0.001000
Epoch 011 | train_loss=0.7739 | val_loss=0.8332 | val_auc=0.7812 | lr=0.001000
Epoch 012 | train_loss=0.7654 | val_loss=0.8459 | val_auc=0.7787 | lr=0.001000
Epoch 013 | train_loss=0.7565 | val_loss=0.8467 | va

In [84]:
model.load_state_dict(
    best_state
)

print(
    "Best validation ROC-AUC:",
    best_auc
)

Best validation ROC-AUC: 0.7812347377108249


# 10. Оценка работы модели

In [85]:
val_loss, val_auc, y_true, y_prob = evaluate(
    model,
    test_loader,
    criterion,
    device
)

y_pred = (
    y_prob >= 0.5
).astype(int)

In [86]:
print(
    "ROC-AUC:",
    roc_auc_score(
        y_true,
        y_prob
    )
)

print(
    "Accuracy:",
    accuracy_score(
        y_true,
        y_pred
    )
)
print(
    "Precision:",
    precision_score(
        y_true,
        y_pred
    )
)
print(
    "Recall:",
    recall_score(
        y_true,
        y_pred
    )
)
print(
    "PR-AUC:",
    average_precision_score(
        y_true,
        y_pred
    )
)

ROC-AUC: 0.7812347377108249
Accuracy: 0.6779067700427999
Precision: 0.4289623174612616
Recall: 0.7680796676962403
PR-AUC: 0.3890279035966436


# 11. Macro Average PR-AUC

In [87]:
ids_test['target'] = y_test

In [88]:
id_to_category = items_human.set_index('id')['category']

def mean_pr_auc(df, preds):
    df = df.copy()
    df['pred'] = np.asarray(preds)

    df['category'] = df['id1'].map(id_to_category)

    scores = {}
    for cat, group in df.groupby('category'):
        scores[cat] = average_precision_score(group['target'].to_numpy(), group['pred'].to_numpy())

    return np.mean(list(scores.values())), scores

In [89]:
score, category_scores = mean_pr_auc(ids_test, y_prob)
print(f"Mean PR-AUC: {score}")

Mean PR-AUC: 0.4527926753554638


In [90]:
len(category_scores)

20

In [91]:
pd.Series(category_scores).sort_values(ascending=False)

Хобби и творчество         0.766357
Детские товары             0.740435
Бытовая химия              0.558270
Красота и гигиена          0.545345
Аптека                     0.544780
Дом и сад                  0.521612
Канцелярские товары        0.483176
Товары для животных        0.481250
Бытовая техника            0.455658
Спорт и отдых              0.441436
Галантерея и аксессуары    0.440579
Продукты питания           0.429846
Строительство и ремонт     0.370979
Обувь                      0.364740
Электроника                0.358827
Музыкальные инструменты    0.339604
Мебель                     0.318523
Ювелирные изделия          0.312068
Одежда                     0.308239
Автотовары                 0.274130
dtype: float64